## Exercise 1: Create the Northbank Financial Data Model

In this exercise, you design the foundational Delta tables for Northbank Financial's analytics platform. The model includes a **customer dimension** designed for SCD Type 2 history tracking, and a **transaction fact table** optimised with liquid clustering and Change Data Feed.

After completing this exercise, return to the lab setup page to explore the tables you created in **Catalog Explorer**.

### Task 1.1 — Create the catalog and schema

Create a Unity Catalog **catalog** named `banking_lab_hugo` with an appropriate comment. Inside it, create a **schema** named `silver`.

In [0]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
# Use the current catalog to reliably find the workspace default catalog,
# regardless of its naming convention.
default_catalog = spark.catalog.currentCatalog()

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS banking_lab_hugo
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Northbank Financial lakehouse platform for customer analytics and regulatory reporting'
""")

Storage root: abfss://unity-catalog-storage@dbstorage2zo6s4666k7uu.dfs.core.windows.net/7405608417148152


DataFrame[]

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS banking_lab_hugo.silver
  COMMENT 'Cleansed and enriched banking data — customer dimensions and transaction facts';

### Task 1.2 — Create the `dim_customer` SCD Type 2 table

Create a managed Delta table `banking_lab_hugo.silver.dim_customer`. This table tracks **customer dimension history** using the SCD Type 2 pattern.

The table must include:

| Column | Type | Notes |
|---|---|---|
| `customer_sk` | `BIGINT GENERATED ALWAYS AS IDENTITY` | Surrogate key — auto-incrementing, unique per version |
| `customer_id` | `STRING NOT NULL` | Business/natural key from source system |
| `full_name` | `STRING` | |
| `email` | `STRING` | |
| `city` | `STRING` | City of primary residence |
| `segment` | `STRING` | Customer segment: Retail, Premium, Business |
| `account_type` | `STRING` | Account type: Current, Savings, Business |
| `valid_from` | `TIMESTAMP NOT NULL` | When this version became effective |
| `valid_to` | `TIMESTAMP NOT NULL` | When this version expired (9999-12-31 for active records) |
| `is_current` | `BOOLEAN` | True for the active version |

Apply **liquid clustering** on `customer_id` and enable **Change Data Feed**.

> 🤖 **Genie Code tip:** Ask:
> *"How do I create a Delta table in Databricks with GENERATED ALWAYS AS IDENTITY, CLUSTER BY, and Change Data Feed enabled in TBLPROPERTIES?"*

**Hints:**
- Use `CLUSTER BY (customer_id)` to optimise lookup performance
- Add `TBLPROPERTIES (delta.enableChangeDataFeed = true)` for audit trail support
- Use `USING DELTA` (this is the recommended format for all tables in Azure Databricks)

In [0]:
%sql
-- TODO: Create banking_lab_hugo.silver.dim_customer as a managed Delta table
-- Include all SCD Type 2 columns, CLUSTER BY (customer_id), and enable Change Data Feed
CREATE TABLE IF NOT EXISTS banking_lab_hugo.silver.dim_customer (
  customer_sk   BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Surrogate key, unique per version',
  customer_id   STRING NOT NULL COMMENT 'Business/natural key from source system',
  full_name     STRING,
  email         STRING,
  city          STRING COMMENT 'City of primary residence',
  segment       STRING COMMENT 'Retail, Premium, Business',
  account_type  STRING COMMENT 'Current, Savings, Business',
  valid_from    TIMESTAMP NOT NULL COMMENT 'When this version became effective',
  valid_to      TIMESTAMP NOT NULL COMMENT 'When this version expired (9999-12-31 for active)',
  is_current    BOOLEAN COMMENT 'True for the active version'
)
USING DELTA
CLUSTER BY (customer_id)
COMMENT 'Customer dimension with SCD Type 2 history'
TBLPROPERTIES (delta.enableChangeDataFeed = true);

### Task 1.3 — Create the `fact_transactions` table

Create a managed Delta table `banking_lab_hugo.silver.fact_transactions` to store individual banking payments.

| Column | Type | Notes |
|---|---|---|
| `transaction_id` | `STRING NOT NULL` | Unique transaction reference |
| `account_id` | `STRING` | Bank account identifier |
| `customer_id` | `STRING` | Links to `dim_customer` |
| `transaction_type` | `STRING` | Credit, Debit, or Transfer |
| `amount` | `DECIMAL(15,2)` | Transaction amount |
| `currency` | `STRING DEFAULT 'GBP'` | Currency code |
| `transaction_date` | `DATE` | Date of the transaction |
| `description` | `STRING` | Transaction narrative |

Apply **liquid clustering** on `customer_id, transaction_date` to support the expected query patterns (filtering by customer and/or date range). Enable **Change Data Feed** for FCA compliance auditing.

> 🤖 **Genie Code tip:** Ask:
> *"How do I create a Delta Lake table in Databricks SQL with liquid clustering and Change Data Feed enabled?"*

**Hints:**
- Use `CLUSTER BY (customer_id, transaction_date)` — you can specify up to 4 clustering keys
- Add `TBLPROPERTIES (delta.enableChangeDataFeed = true)`
- Use `STRING DEFAULT 'GBP'` for the currency column default value

In [0]:
%sql
-- TODO: Create banking_lab_hugo.silver.fact_transactions as a managed Delta table
-- Apply CLUSTER BY (customer_id, transaction_date) and enable Change Data Feed
CREATE TABLE IF NOT EXISTS banking_lab_hugo.silver.fact_transactions (
  transaction_id    STRING NOT NULL COMMENT 'Unique transaction reference',
  account_id        STRING COMMENT 'Bank account identifier',
  customer_id       STRING COMMENT 'Links to dim_customer',
  transaction_type  STRING COMMENT 'Credit, Debit, or Transfer',
  amount            DECIMAL(15,2) COMMENT 'Transaction amount',
  currency          STRING DEFAULT 'GBP' COMMENT 'Currency code',
  transaction_date  DATE COMMENT 'Date of the transaction',
  description       STRING COMMENT 'Transaction narrative'
)
USING DELTA
CLUSTER BY (customer_id, transaction_date)
COMMENT 'Banking payment transactions fact table'
TBLPROPERTIES (
  delta.enableChangeDataFeed = true,
  'delta.feature.allowColumnDefaults' = 'supported'
);

## Exercise 2: Implement SCD Type 2 for the Customer Dimension

Northbank's regulatory reporting must reflect the **customer profile at the time of each transaction**, not just the current state. In this exercise, you load initial customer records, apply realistic attribute changes, and implement the **two-step MERGE + INSERT pattern** for SCD Type 2. You then query full customer history and perform a point-in-time lookup.

### Task 2.1 — Load the initial customer records

The cell below defines the **initial dataset** for 8 Northbank customers as a PySpark DataFrame. Your task is to **write this DataFrame to `banking_lab_hugo.silver.dim_customer`** in append mode.

> 🤖 **Genie Code tip:** Ask:
> *"How do I write a PySpark DataFrame to an existing Delta table in Databricks using saveAsTable in append mode?"*

**Hints:**
- Use `df.write.mode("append").saveAsTable("banking_lab_hugo.silver.dim_customer")`
- The data and DataFrame setup are already provided — add the write statement after the DataFrame is built

In [0]:
from pyspark.sql.functions import lit, to_timestamp

initial_customers = [
    ("C-1001", "Emma Hartley",      "emma.hartley@northbank.com",     "London",     "Retail",   "Savings"),
    ("C-1002", "James Weston",      "james.weston@northbank.com",     "Manchester", "Retail",   "Current"),
    ("C-1003", "Sophia Chen",       "sophia.chen@northbank.com",      "Edinburgh",  "Premium",  "Savings"),
    ("C-1004", "Oliver Banks",      "oliver.banks@northbank.com",     "Birmingham", "Retail",   "Current"),
    ("C-1005", "Aisha Patel",       "aisha.patel@northbank.com",      "London",     "Business", "Business"),
    ("C-1006", "Liam Murray",       "liam.murray@northbank.com",      "Leeds",      "Retail",   "Savings"),
    ("C-1007", "Charlotte Wright",  "charlotte.wright@northbank.com", "Bristol",    "Premium",  "Current"),
    ("C-1008", "Noah Thompson",     "noah.thompson@northbank.com",    "Glasgow",    "Retail",   "Savings"),
]

df_initial = spark.createDataFrame(
    initial_customers,
    ["customer_id", "full_name", "email", "city", "segment", "account_type"]
)

df_initial = (
    df_initial
    .withColumn("valid_from",  to_timestamp(lit("2020-01-01 00:00:00")))
    .withColumn("valid_to",    to_timestamp(lit("9999-12-31 00:00:00")))
    .withColumn("is_current",  lit(True))
)

# TODO: Write df_initial to banking_lab_hugo.silver.dim_customer (append mode)
# customer_sk is an identity column, so it is generated automatically
df_initial.write.mode("append").saveAsTable("banking_lab_hugo.silver.dim_customer")
print(f"Loaded {df_initial.count()} customers into banking_lab_hugo.silver.dim_customer")

Loaded 8 customers into banking_lab_hugo.silver.dim_customer


In [0]:
%sql
-- Verify: you should see 8 customer records
SELECT customer_sk, customer_id, full_name, city, segment, is_current
FROM banking_lab_hugo.silver.dim_customer
ORDER BY customer_sk;

customer_sk,customer_id,full_name,city,segment,is_current
1,C-1001,Emma Hartley,London,Retail,true
2,C-1002,James Weston,Manchester,Retail,true
3,C-1003,Sophia Chen,Edinburgh,Premium,true
4,C-1004,Oliver Banks,Birmingham,Retail,true
5,C-1005,Aisha Patel,London,Business,true
6,C-1006,Liam Murray,Leeds,Retail,true
7,C-1007,Charlotte Wright,Bristol,Premium,true
8,C-1008,Noah Thompson,Glasgow,Retail,true


### Task 2.2 — Load staging data with customer changes

The operations team has received the following change notifications:

| Customer | Change |
|---|---|
| **C-1001 Emma Hartley** | Moved from **London → Oxford**, upgraded to **Premium** segment |
| **C-1003 Sophia Chen** | Relocated from **Edinburgh → London** |
| **C-1006 Liam Murray** | Updated email address only (city and segment unchanged) |
| **C-1009 Priya Singh** | **New customer** joining from Cardiff, Retail segment |

Run the cell below to create a staging DataFrame and register it as a temp view. You will use it in the next task.

In [0]:
staging_data = [
    ("C-1001", "Emma Hartley",    "emma.hartley@northbank.com",     "Oxford",   "Premium",  "Savings"),
    ("C-1003", "Sophia Chen",     "sophia.chen@northbank.com",      "London",   "Premium",  "Savings"),
    ("C-1006", "Liam Murray",     "liam.new@northbank.com",         "Leeds",    "Retail",   "Savings"),
    ("C-1009", "Priya Singh",     "priya.singh@northbank.com",      "Cardiff",  "Retail",   "Current"),
]

df_staging = spark.createDataFrame(
    staging_data,
    ["customer_id", "full_name", "email", "city", "segment", "account_type"]
)
df_staging.createOrReplaceTempView("staging_customers")
print("Staging data loaded. Temp view 'staging_customers' is available.")
display(df_staging)

Staging data loaded. Temp view 'staging_customers' is available.


customer_id,full_name,email,city,segment,account_type
C-1001,Emma Hartley,emma.hartley@northbank.com,Oxford,Premium,Savings
C-1003,Sophia Chen,sophia.chen@northbank.com,London,Premium,Savings
C-1006,Liam Murray,liam.new@northbank.com,Leeds,Retail,Savings
C-1009,Priya Singh,priya.singh@northbank.com,Cardiff,Retail,Current


### Task 2.3 — Apply SCD Type 2: Step 1 — Close changed records

Write a `MERGE INTO` statement that compares `staging_customers` against the current versions in `dim_customer`. For any record where `city`, `segment`, `email`, or `account_type` has changed, **close the current version** by setting:
- `valid_to = current_timestamp()`
- `is_current = false`

> 🤖 **Genie Code tip:** Ask:
> *"Write a Databricks SQL MERGE statement that closes SCD Type 2 dimension records by setting is_current=false and valid_to=current_timestamp() when any tracked attribute changes."*

**Hints:**
- Match on `target.customer_id = source.customer_id AND target.is_current = true`
- The `WHEN MATCHED AND (...)` condition should check if **any** of the tracked attributes differ
- Only update `valid_to` and `is_current` — do not overwrite the historical data columns

In [0]:
%sql
-- Step 1: Close the current version of changed customer records
-- MERGE INTO banking_lab_hugo.silver.dim_customer ...
-- TODO: Write the MERGE statement
MERGE INTO banking_lab_hugo.silver.dim_customer AS target
USING staging_customers AS source
  ON target.customer_id = source.customer_id
 AND target.is_current = true
WHEN MATCHED AND (
     NOT (target.city         <=> source.city)
  OR NOT (target.segment      <=> source.segment)
  OR NOT (target.email        <=> source.email)
  OR NOT (target.account_type <=> source.account_type)
) THEN UPDATE SET
  target.valid_to   = current_timestamp(),
  target.is_current = false;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
3,3,0,0


### Task 2.3 — Apply SCD Type 2: Step 2 — Insert new versions and new customers

Write an `INSERT INTO` statement to:
1. Add **new current rows** for customers whose old versions were just closed (C-1001, C-1003, C-1006)
2. Add the **brand-new customer** C-1009 who has no existing record

New rows should have `valid_from = current_timestamp()`, `valid_to = TIMESTAMP '9999-12-31'`, and `is_current = true`.

> 🤖 **Genie Code tip:** Ask:
> *"Write a Databricks SQL INSERT statement to add new current SCD Type 2 rows from a staging table, only for customers where no is_current=true record exists."*

**Hint:** Use a `WHERE NOT EXISTS (SELECT 1 FROM banking_lab_hugo.silver.dim_customer d WHERE d.customer_id = s.customer_id AND d.is_current = true)` to identify which staging records need a new row inserted.

In [0]:
%sql
-- Step 2: Insert new current versions for changed customers and the new customer
-- INSERT INTO banking_lab_hugo.silver.dim_customer ...
-- TODO: Write the INSERT statement
INSERT INTO banking_lab_hugo.silver.dim_customer
  (customer_id, full_name, email, city, segment, account_type, valid_from, valid_to, is_current)
SELECT
  s.customer_id, s.full_name, s.email, s.city, s.segment, s.account_type,
  current_timestamp()              AS valid_from,
  TIMESTAMP '9999-12-31 00:00:00'  AS valid_to,
  true                             AS is_current
FROM staging_customers s
WHERE NOT EXISTS (
  SELECT 1
  FROM banking_lab_hugo.silver.dim_customer d
  WHERE d.customer_id = s.customer_id
    AND d.is_current = true
);

num_affected_rows,num_inserted_rows
4,4


### Task 2.4 — Query the full version history of a customer

Write a query to show **all versions** of customer `C-1001` (Emma Hartley), ordered by `valid_from`. You should see:
- Version 1: London / Retail (expired)
- Version 2: Oxford / Premium (current)

> 🤖 **Genie Code tip:** Ask:
> *"How do I query all SCD Type 2 versions for a specific customer in a Databricks Delta table?"*

**Hint:** Filter on `customer_id = 'C-1001'` and `ORDER BY valid_from ASC`.

In [0]:
%sql
-- TODO: Show all versions of customer C-1001, ordered by valid_from
SELECT customer_sk, customer_id, full_name, email, city, segment, account_type,
       valid_from, valid_to, is_current
FROM banking_lab_hugo.silver.dim_customer
WHERE customer_id = 'C-1001'
ORDER BY valid_from ASC;

customer_sk,customer_id,full_name,email,city,segment,account_type,valid_from,valid_to,is_current
1,C-1001,Emma Hartley,emma.hartley@northbank.com,London,Retail,Savings,2020-01-01T00:00:00.000Z,2026-11-10T13:04:14.033Z,false
9,C-1001,Emma Hartley,emma.hartley@northbank.com,Oxford,Premium,Savings,2026-11-10T13:04:26.602Z,9999-12-31T00:00:00.000Z,true


### Task 2.5 — Point-in-time customer lookup

Northbank's regulatory reporting team needs to know the **customer profile as it existed on 2020-06-15** — before any of the changes above were applied.

Write a query that returns one row per customer showing their profile on that date. Use the `valid_from` and `valid_to` columns to select the version that was active on 2020-06-15.

> 🤖 **Genie Code tip:** Ask:
> *"How do I write a point-in-time query on an SCD Type 2 table using valid_from and valid_to date range filtering?"*

**Hint:** A version was active on date `D` when `valid_from <= D AND valid_to > D`. Expected result: all 8 original customers in their initial state (no Oxford, no London for Sophia).

In [0]:
%sql
-- TODO: Return the customer profile as it existed on 2020-06-15
-- Expected: 8 rows showing the original city and segment values
SELECT customer_id, full_name, email, city, segment, account_type, valid_from, valid_to
FROM banking_lab_hugo.silver.dim_customer
WHERE valid_from <= TIMESTAMP '2020-06-15 00:00:00'
  AND valid_to   >  TIMESTAMP '2020-06-15 00:00:00'
ORDER BY customer_id;

customer_id,full_name,email,city,segment,account_type,valid_from,valid_to
C-1001,Emma Hartley,emma.hartley@northbank.com,London,Retail,Savings,2020-01-01T00:00:00.000Z,2026-11-10T13:04:14.033Z
C-1002,James Weston,james.weston@northbank.com,Manchester,Retail,Current,2020-01-01T00:00:00.000Z,9999-12-31T00:00:00.000Z
C-1003,Sophia Chen,sophia.chen@northbank.com,Edinburgh,Premium,Savings,2020-01-01T00:00:00.000Z,2026-11-10T13:04:14.033Z
C-1004,Oliver Banks,oliver.banks@northbank.com,Birmingham,Retail,Current,2020-01-01T00:00:00.000Z,9999-12-31T00:00:00.000Z
C-1005,Aisha Patel,aisha.patel@northbank.com,London,Business,Business,2020-01-01T00:00:00.000Z,9999-12-31T00:00:00.000Z
C-1006,Liam Murray,liam.murray@northbank.com,Leeds,Retail,Savings,2020-01-01T00:00:00.000Z,2026-11-10T13:04:14.033Z
C-1007,Charlotte Wright,charlotte.wright@northbank.com,Bristol,Premium,Current,2020-01-01T00:00:00.000Z,9999-12-31T00:00:00.000Z
C-1008,Noah Thompson,noah.thompson@northbank.com,Glasgow,Retail,Savings,2020-01-01T00:00:00.000Z,9999-12-31T00:00:00.000Z


## Exercise 3: Change Data Feed & FCA Audit Trail

Basel III and FCA regulations require Northbank to maintain a **queryable audit trail** of all modifications to transaction data. In this exercise, you load payment transactions, simulate data corrections (a common real-world event), and then query the **Change Data Feed** to produce a full audit log.

### Task 3.1 — Load initial transactions

Run the cell below to insert 10 payment transactions into `fact_transactions`. These represent one day's batch of processed payments.

In [0]:
%sql
INSERT INTO banking_lab_hugo.silver.fact_transactions VALUES
  ('TXN-001', 'ACC-001', 'C-1001', 'Credit',  1500.00, 'GBP', '2025-11-01', 'Salary payment'),
  ('TXN-002', 'ACC-001', 'C-1001', 'Debit',    120.50, 'GBP', '2025-11-03', 'Amazon purchase'),
  ('TXN-003', 'ACC-002', 'C-1002', 'Debit',    250.00, 'GBP', '2025-11-05', 'Grocery store'),
  ('TXN-004', 'ACC-003', 'C-1003', 'Credit',  3200.00, 'GBP', '2025-11-07', 'Salary payment'),
  ('TXN-005', 'ACC-003', 'C-1003', 'Transfer', 500.00, 'GBP', '2025-11-10', 'Rent transfer'),
  ('TXN-006', 'ACC-004', 'C-1004', 'Debit',     89.99, 'GBP', '2025-11-12', 'Utility bill'),
  ('TXN-007', 'ACC-005', 'C-1005', 'Debit',   1500.00, 'GBP', '2025-11-14', 'Business expense'),
  ('TXN-008', 'ACC-006', 'C-1006', 'Credit',  2800.00, 'GBP', '2025-11-15', 'Salary payment'),
  ('TXN-009', 'ACC-007', 'C-1007', 'Debit',    340.00, 'GBP', '2025-11-18', 'Car insurance'),
  ('TXN-010', 'ACC-008', 'C-1008', 'Transfer', 200.00, 'GBP', '2025-11-20', 'Transfer to savings');

num_affected_rows,num_inserted_rows
10,10


### Task 3.2 — Simulate transaction corrections

Northbank's reconciliation team identified two amount discrepancies. Run the cell below to apply the corrections.

In [0]:
%sql
-- Correction 1: TXN-003 amount was mis-keyed (correct: £275.50, not £250.00)
UPDATE banking_lab_hugo.silver.fact_transactions
SET amount = 275.50, description = 'Grocery store (corrected)'
WHERE transaction_id = 'TXN-003';

-- Correction 2: TXN-007 amount is under dispute
UPDATE banking_lab_hugo.silver.fact_transactions
SET amount = 1450.00, description = 'Business expense (disputed amount)'
WHERE transaction_id = 'TXN-007';

num_affected_rows
1


### Task 3.3 — Query the Change Data Feed to build an audit trail

Write a query using `table_changes()` to retrieve **all change events** recorded for `fact_transactions` since the beginning of its history. Include the `_change_type`, `_commit_version`, and `_commit_timestamp` metadata columns.

> 🤖 **Genie Code tip:** Ask:
> *"How do I query the Change Data Feed on a Delta table in Databricks using table_changes()? Show change type and commit metadata columns."*

**Hint:** Use `table_changes('banking_lab_hugo.silver.fact_transactions', 0)` — the second argument is the starting version number.

In [0]:
%sql
-- TODO: Query all changes to fact_transactions from version 0 onwards
-- Include: transaction_id, amount, description, _change_type, _commit_version, _commit_timestamp
-- ORDER BY _commit_timestamp, transaction_id
SELECT transaction_id, amount, description,
       _change_type, _commit_version, _commit_timestamp
FROM table_changes('banking_lab_hugo.silver.fact_transactions', 0)
ORDER BY _commit_timestamp, transaction_id;

transaction_id,amount,description,_change_type,_commit_version,_commit_timestamp
TXN-001,1500.00,Salary payment,insert,1,2026-11-10T13:04:53.000Z
TXN-002,120.50,Amazon purchase,insert,1,2026-11-10T13:04:53.000Z
TXN-003,250.00,Grocery store,insert,1,2026-11-10T13:04:53.000Z
TXN-004,3200.00,Salary payment,insert,1,2026-11-10T13:04:53.000Z
TXN-005,500.00,Rent transfer,insert,1,2026-11-10T13:04:53.000Z
TXN-006,89.99,Utility bill,insert,1,2026-11-10T13:04:53.000Z
TXN-007,1500.00,Business expense,insert,1,2026-11-10T13:04:53.000Z
TXN-008,2800.00,Salary payment,insert,1,2026-11-10T13:04:53.000Z
TXN-009,340.00,Car insurance,insert,1,2026-11-10T13:04:53.000Z
TXN-010,200.00,Transfer to savings,insert,1,2026-11-10T13:04:53.000Z


### Task 3.4 — Isolate corrected records for FCA reporting

For FCA reporting, the compliance team needs a view showing only the **post-correction state** of modified transactions. Filter the Change Data Feed to show only `update_postimage` records.

> 🤖 **Genie Code tip:** Ask:
> *"How do I filter table_changes() results in Databricks to show only update_postimage records?"*

**Hint:** Each `UPDATE` operation generates two rows: `update_preimage` (before) and `update_postimage` (after the change). Filter for `_change_type = 'update_postimage'`. Expected result: TXN-003 with £275.50 and TXN-007 with £1450.00.

In [0]:
%sql
-- TODO: Show only the post-correction state of updated transactions
-- Filter table_changes() results for _change_type = 'update_postimage'
SELECT transaction_id, amount, description,
       _change_type, _commit_version, _commit_timestamp
FROM table_changes('banking_lab_hugo.silver.fact_transactions', 0)
WHERE _change_type = 'update_postimage'
ORDER BY transaction_id;

transaction_id,amount,description,_change_type,_commit_version,_commit_timestamp
TXN-003,275.50,Grocery store (corrected),update_postimage,2,2026-11-10T17:17:35.000Z
TXN-007,1450.00,Business expense (disputed amount),update_postimage,3,2026-11-10T17:17:40.000Z


## Exercise 4: Delta Lake Time Travel

Delta Lake maintains a full **transaction log** of every change made to a table. This gives you the ability to query **previous versions** of your data and even **restore** a table to an earlier state — without restoring from backup. This is critical for Northbank when transaction data is accidentally modified.

### Task 4.1 — Inspect the transaction history

Use `DESCRIBE HISTORY` to view the complete operation log of `fact_transactions`. The output shows every version of the table, including the operation type, timestamp, and user.

> 🤖 **Genie Code tip:** Ask:
> *"How do I view the full operation history of a Delta table in Databricks, including all version numbers?"*

Note the **version numbers** in the output — you will use them in the next tasks.

In [0]:
%sql
-- TODO: Show the full operation history of banking_lab_hugo.silver.fact_transactions
DESCRIBE HISTORY banking_lab_hugo.silver.fact_transactions;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-11-10T17:17:43.000Z,145297173496900,hugo.alcaide@tajamar365.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3406120736654009),de1d3130-25eb-4ab7-95bb-9b92cb505de9,1110-171712-wsa7d4vi-v2n,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 11104, p25FileSize -> 4124, numDeletionVectorsRemoved -> 1, minFileSize -> 4124, numAddedFiles -> 1, maxFileSize -> 4124, p75FileSize -> 4124, p50FileSize -> 4124, numAddedBytes -> 4124)",null,Databricks-Runtime/19.14.x-photon-scala2.13
3,2026-11-10T17:17:40.000Z,145297173496900,hugo.alcaide@tajamar365.com,UPDATE,"Map(predicate -> [""(transaction_id#11463 = TXN-007)""])",null,List(3406120736654009),de1d3130-25eb-4ab7-95bb-9b92cb505de9,1110-171712-wsa7d4vi-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 1, executionTimeMs -> 3520, numDeletionVectorsUpdated -> 1, scanTimeMs -> 710, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 3968, rewriteTimeMs -> 2809)",null,Databricks-Runtime/19.14.x-photon-scala2.13
2,2026-11-10T17:17:35.000Z,145297173496900,hugo.alcaide@tajamar365.com,UPDATE,"Map(predicate -> [""(transaction_id#10127 = TXN-003)""])",null,List(3406120736654009),6b2387ba-7a18-428c-928e-b2297b2acce5,1110-171712-wsa7d4vi-v2n,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 1, executionTimeMs -> 10841, numDeletionVectorsUpdated -> 0, scanTimeMs -> 4783, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 3923, rewriteTimeMs -> 6020)",null,Databricks-Runtime/19.14.x-photon-scala2.13
1,2026-11-10T13:04:53.000Z,145297173496900,hugo.alcaide@tajamar365.com,WRITE,"Map(partitionBy -> [], clusterBy -> [""customer_id"",""transaction_date""], statsOnLoad -> true, mode -> Append, clusteringOnWriteStatus -> Reason for skipping: rowInsertionCountMismatch)",null,List(3406120736654009),44694b92-7ed2-49df-8ef0-9c83963c211a,1110-125836-roha5keq-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 10, numOutputBytes -> 3213)",null,Databricks-Runtime/19.14.x-photon-scala2.13
0,2026-11-10T13:03:39.000Z,145297173496900,hugo.alcaide@tajamar365.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [""customer_id"",""transaction_date""], description -> Banking payment transactions fact table, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableChangeDataFeed"":""true"",""delta.enableDeletionVectors"":""true"",""databricks.delta.expressionStats.selectedColumns"":""upper(customer_id),lower(customer_id)"",""databricks.internal.autoUpgrades.delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""b188ccd4-3c22-41ae-9861-b4e407f4b261"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-e5f55b7e-3989-4f41-a9bc-86a1172774f3"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-547b3b56-4262-4d7d-a205-68f3be9f2330""}, statsOnLoad -> false)",null,List(3406120736654009),7d5bd2ee-e4f1-4318-82ea-6737a49612f1,1110-125836-roha5keq-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.14.x-photon-scala2.13


### Task 4.2 — Query original transaction amounts before corrections

Using `VERSION AS OF`, retrieve `TXN-003` and `TXN-007` **as they appeared after the initial INSERT but before the corrections** were applied.

> 🤖 **Genie Code tip:** Ask:
> *"How do I query a previous version of a Delta table in Databricks using VERSION AS OF in a SELECT statement?"*

**Hint:** The initial INSERT corresponds to version **1** (version 0 is the table creation). Expected result: TXN-003 with £250.00 and TXN-007 with £1500.00.

In [0]:
%sql
-- TODO: Query fact_transactions VERSION AS OF 1
-- Show transaction_id, amount, description for TXN-003 and TXN-007
SELECT transaction_id, amount, description
FROM banking_lab_hugo.silver.fact_transactions VERSION AS OF 1
WHERE transaction_id IN ('TXN-003', 'TXN-007')
ORDER BY transaction_id;

transaction_id,amount,description
TXN-003,250.00,Grocery store
TXN-007,1500.00,Business expense


### Task 4.3 — Restore the table to its pre-correction state

The reconciliation team has confirmed the corrections were applied in error and must be rolled back. Use `RESTORE TABLE` to revert `fact_transactions` to **version 1** (after the initial load, before any corrections).

After restoring, verify that TXN-003 and TXN-007 show their original amounts.

> 🤖 **Genie Code tip:** Ask:
> *"How do I use RESTORE TABLE in Databricks to roll back a Delta table to a specific version number?"*

**Hint:** Use `RESTORE TABLE banking_lab_hugo.silver.fact_transactions TO VERSION AS OF 1;`

In [0]:
%sql
-- TODO: Restore fact_transactions to VERSION AS OF 1
RESTORE TABLE banking_lab_hugo.silver.fact_transactions TO VERSION AS OF 1;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
3213,1,1,1,4124,3213


In [0]:
%sql
-- TODO: Verify the original amounts are restored for TXN-003 and TXN-007
-- Expected: TXN-003 = £250.00, TXN-007 = £1500.00
SELECT transaction_id, amount, description
FROM banking_lab_hugo.silver.fact_transactions
WHERE transaction_id IN ('TXN-003', 'TXN-007')
ORDER BY transaction_id;

transaction_id,amount,description
TXN-003,250.00,Grocery store
TXN-007,1500.00,Business expense
